# phase 2: multimodal rag (mRAG)

extends the text rag from notebook 1 into a vision-language pipeline.  
the handbook pages are converted to images

In [ ]:
import sys
!{sys.executable} -m pip install pymupdf pillow torch transformers chromadb \
    langsmith langchain-ollama ollama --quiet

In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")  # set your own LangSmith key
os.environ["LANGCHAIN_PROJECT"] = "EndTerm_mRAG"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

In [3]:
PDF_PATH        = "studentHandbook.pdf"
IMAGE_DIR       = "./page_images"
CLIP_MODEL_ID   = "openai/clip-vit-base-patch32"  # 512-dim
# VLM_MODEL       = "moondream:1.8b"
VLM_MODEL       = "llava-phi3:3.8b"
CLIP_PERSIST_DIR = "./chroma_clip_db"
TOP_K           = 2

In [4]:
# convert every page of the pdf to a png image using pymupdf
# dpi=150 is the sweet spot: readable for the vlm but not huge files
import pymupdf

os.makedirs(IMAGE_DIR, exist_ok=True)

doc = pymupdf.open(PDF_PATH)
image_paths = []

for i, page in enumerate(doc):
    pix  = page.get_pixmap(dpi=150)
    path = os.path.join(IMAGE_DIR, f"page_{i+1:03d}.png")
    pix.save(path)
    image_paths.append(os.path.abspath(path))

print(f"saved {len(image_paths)} page images to {IMAGE_DIR}")

saved 30 page images to ./page_images


In [5]:
import torch
from transformers import CLIPModel, CLIPProcessor

clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model.eval()
print("clip model loaded")

clip model loaded


In [6]:
from PIL import Image

def get_image_embedding(image_path: str) -> list:
    image  = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        feats = clip_model.get_image_features(**inputs)
    if not isinstance(feats, torch.Tensor):
        feats = feats.pooler_output
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats.squeeze().tolist()


def get_text_embedding(query: str) -> list:
    inputs = clip_processor(text=query, return_tensors="pt", padding=True)
    with torch.no_grad():
        feats = clip_model.get_text_features(**inputs)
    if not isinstance(feats, torch.Tensor):
        feats = feats.pooler_output
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats.squeeze().tolist()

In [7]:
# using raw chroma client (not langchain wrapper) to have full control over 512-dim clip embeddings
import chromadb

chroma_client = chromadb.PersistentClient(path=CLIP_PERSIST_DIR)
collection    = chroma_client.get_or_create_collection(
    name="handbook_pages",
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    print("indexing pages...")
    for i, img_path in enumerate(image_paths):
        emb = get_image_embedding(img_path)
        collection.add(
            ids=[f"page_{i+1}"],
            embeddings=[emb],
            metadatas=[{"image_path": img_path, "page_num": i + 1}]
        )
        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(image_paths)} done")
    print(f"indexed {collection.count()} pages")
else:
    print(f"collection already has {collection.count()} pages, skipping indexing")

indexing pages...
  5/30 done
  10/30 done
  15/30 done
  20/30 done
  25/30 done
  30/30 done
indexed 30 pages


In [8]:
def retrieve_pages(query: str, k: int = TOP_K) -> list:
    query_emb = get_text_embedding(query)
    results   = collection.query(query_embeddings=[query_emb], n_results=k)
    return [m["image_path"] for m in results["metadatas"][0]]

In [9]:
import ollama


def generate_mrag_answer(query: str, image_paths: list) -> str:
    response = ollama.chat(
        model=VLM_MODEL,
        messages=[{
            "role": "user",
            "content": f"answer the following question using only the document pages shown. if the answer is not visible in the pages, say exactly: i don't know\n\nquestion: {query}",
            "images": image_paths
        }],
        options={"num_ctx": 4096}
    )
    return response["message"]["content"]

In [10]:
def predict_mrag(inputs: dict) -> dict:
    query       = inputs["question"]
    pages       = retrieve_pages(query)
    answer      = generate_mrag_answer(query, pages)
    return {"answer": answer}

In [11]:
# reuse the exact same dataset and judge model so results are directly comparable to notebook 1
from langchain_ollama import ChatOllama
from langsmith.schemas import Example, Run
from langsmith.evaluation import evaluate

dataset_name = "Handbook_QA_Test"


def local_correctness_evaluator(run: Run, example: Example) -> dict:
    predicted = run.outputs.get("answer", "")
    expected  = example.outputs.get("answer", "")
    question  = example.inputs.get("question", "")

    judge  = ChatOllama(model="mistral:latest", temperature=0)
    prompt = f"""you are a grading assistant. compare the predicted answer to the reference answer.

question: {question}
reference: {expected}
predicted: {predicted}

is the predicted answer correct? it is correct if it captures the key facts, even if worded differently.
reply with only CORRECT or INCORRECT."""

    result = judge.invoke(prompt)
    score = 1 if result.content.strip().upper() == "CORRECT" else 0
    return {"key": "correctness", "score": score}


results_mrag = evaluate(
    predict_mrag,
    data=dataset_name,
    evaluators=[local_correctness_evaluator],
    experiment_prefix="mrag-Llava-phi3-handbook"
)

View the evaluation results for experiment: 'mrag-Llava-phi3-handbook-2fa1fcb3' at:
https://smith.langchain.com/o/efb57b8e-29aa-4df1-a914-019fa7fd3d4f/datasets/4596fae7-2a00-4e07-8cfe-5fdcef371459/compare?selectedSessions=57febf3d-1144-47f4-b5c7-00e2b5322840




In [13]:
import pandas as pd

mrag_df    = results_mrag.to_pandas()
mrag_score = mrag_df["feedback.correctness"].mean()
print(f"mRAG (moondream) correctness: {mrag_score:.2f}")
mrag_df[["inputs.question", "outputs.answer", "feedback.correctness"]]

mRAG (moondream) correctness: 0.30


,inputs.question,outputs.answer,feedback.correctness
0,what is the current stock price of infosys on ...,Current NSE Stock Price for Infosys Limited: R...,1
1,is the campus doctor free to visit and what ar...,\nThe document page provides information on th...,0
2,are microwaves or electric kettles allowed in ...,\nI don't know.,0
3,how do i apply for leave to go home for a week...,"\nTo apply for leave to go home for a weekend,...",0
4,how much wifi data do i get per month and what...,i dont have enough information to answer your ...,0
5,how does the hostel laundry service work and h...,\n1. The Hostel Laundry Service works by picki...,0
6,can i order zomato or swiggy and eat it in my ...,\nThe image you provided does not contain any ...,0
7,can i cook food in my hostel room?,Hostel Kitchen : Cooking & Dining (Hindi Edition),0
8,can i come back after 9pm on a friday night?,I don't have enough information to answer your...,1
9,what time do i have to be back in the hostel b...,\n[Your Name],1


### mRAG failure analysis

**failure 1: clip was not trained on document pages**

CLIP was trained on natural image-caption pairs (LAION dataset). dense text pages all look visually similar to CLIP -- walls of text with no distinguishing visual features. for a question about curfew timings, CLIP might retrieve pages 5, 12, and 20 instead of page 9 where section 15.1 actually lives. the text encoder and image encoder live in the same embedding space but that space was not designed for document retrieval. this is retrieval-side failure.

**failure 2: moondream struggles with dense small-font text**

even when the right page is retrieved, moondream:1.8b has limited OCR capability. the student handbook pages have small serif fonts, multi-column layouts, and dense regulatory text. moondream's internal resolution downsamples images significantly, making fine text illegible. on questions that require reading a specific numbered subsection (e.g. 27.7.7 about the 3kg laundry limit), the VLM often returns a generic or partially hallucinated answer.

**failure 3: visual token budget is finite**

moondream processes each image at a fixed internal resolution. feeding 3 full pages means the per-page detail is further compressed. a text-only RAG chunk is lossless -- every character is preserved. an image-based representation at 1.8B params is fundamentally lossy for text-heavy documents.

**why text rag beats mRAG here**

the handbook is a text-native pdf. every character can be extracted perfectly with pymupdf. nomic-embed-text is purpose-built for semantic text retrieval and outperforms CLIP on document search tasks by a large margin. mRAG's advantage comes in documents with rich visual content (diagrams, photos, handwritten notes). for a policy document like a student handbook, text rag with a sufficiently capable generator (llama3.2:3b) is the clearly superior architecture.